# Part 2: Cross-Domain Acne Classification

Train a binary acne / non-acne classifier on **ACNE04 patches**, then evaluate on the **DermNet** test set with three increasingly aggressive domain adaptation strategies:

1. **Baseline** (no adaptation)
2. **+ Heavy training-time augmentation** (already baked into the trained model)
3. **+ Reinhard color normalization** at test time
4. **+ TTA** (horizontal flip averaging)

Plus Grad-CAM visualizations of 10 DermNet predictions.

## 0. Setup

In [ ]:
import os
if not os.path.exists('acne_yang_project'):
    !git clone https://github.com/<YOUR_USERNAME>/acne_yang_project.git
%cd acne_yang_project
!pip install -q -r requirements.txt

## 1. Generate ACNE04 patches

In [ ]:
from part2_classification.make_patches import make_patches
counts = make_patches(
    coco_root='data/acne04/coco',
    out_root='data/acne04_patches',
    expand=1.3, target_size=224, neg_per_pos=1.0,
)
counts

## 2. Download DermNet (Kaggle)

Upload your Kaggle API token (`kaggle.json`) before running this cell.

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d shubhamgoel27/dermnet -p data/dermnet --unzip
!ls data/dermnet

## 3. Train classifier on ACNE04 (with heavy augmentation)

In [ ]:
from part2_classification.train_classifier import train as train_clf
best_weights = train_clf(
    patches_root='data/acne04_patches',
    out_dir='outputs/classifier',
    backbone='resnet50',
    epochs=15,
    heavy_da=True,
)
best_weights

## 4. Domain-gap ablation on DermNet

In [ ]:
from part2_classification.evaluate_dermnet import evaluate as eval_dermnet
import pandas as pd

table = {}
for da, tta, name in [
    ('none', False, 'baseline'),
    ('histogram', False, '+histogram'),
    ('reinhard', False, '+reinhard'),
    ('reinhard', True, '+reinhard+TTA'),
]:
    table[name] = eval_dermnet(
        weights=str(best_weights),
        dermnet_root='data/dermnet',
        da_method=da,
        tta=tta,
        out_dir=f'outputs/dermnet_eval/{name.replace("+","").replace(" ","_")}',
    )
pd.DataFrame(table).T[['acc', 'f1', 'auroc']]

## 5. Grad-CAM on 10 DermNet predictions

In [ ]:
from part2_classification.gradcam import visualize_gradcam
from IPython.display import Image as IPyImage

out = visualize_gradcam(
    weights=str(best_weights),
    predictions_json='outputs/dermnet_eval/reinhardTTA/predictions.json',
    dermnet_train_root='data/dermnet/train',
    da_method='reinhard',
    n_samples=10,
)
IPyImage(filename=str(out))